# SQL for Data Extraction and Analysis

## Objective

The objective of this task is to use SQL for data extraction and business analysis on the cleaned IBM Telco Customer Churn dataset.

The project covers SQL fundamentals, advanced SQL concepts, and integration of SQL with Python to generate business insights.

-------------------------------------
## Import Libraries

In [1]:
import pandas as pd
import sqlite3

----------------------------
## Database Creation

The cleaned dataset is loaded into SQLite and stored as a database table for SQL-based analysis.

In [2]:
df = pd.read_csv('../data/processed/telco_cleaned.csv')

In [3]:
conn = sqlite3.connect('../data/telco.db')

In [4]:
df.to_sql("customers",conn,if_exists="replace",index = False)

7043

In [5]:
pd.read_sql(
    "SELECT * FROM customers LIMIT 5",
    conn
)

,CustomerID,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,...,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,No,...,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,86,3239,Competitor made better offer
1,9237-HQITU,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,Yes,...,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,67,2701,Moved
2,9305-CDSKC,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,Yes,...,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,Yes,86,5372,Moved
3,7892-POOKP,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,Yes,...,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,84,5003,Moved
4,0280-XJGEX,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,Yes,...,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,Yes,89,5340,Competitor had better devices


### Observation

The cleaned customer dataset was successfully imported into SQLite and stored in the `customers` table for analysis.

----------------------------

# Business Questions & Insights

### Business Question 1 :

How many customers are present in the dataset?

In [6]:
pd.read_sql("SELECT COUNT(*) FROM customers",conn)

,COUNT(*)
0,7043


### Insight

The dataset contains 7043 customer records available for analysis.

---------------------------------

### Business Question 2 :

How many customers have churned?

In [7]:
pd.read_sql("SELECT COUNT([Churn Reason]) FROM customers",conn)

,COUNT([Churn Reason])
0,1869


### Insight:

There 1869 customers who have churned.

-----------------------------------------------------

### Business Question 3 :

Which contract type is most commonly used by customers?

In [8]:
pd.read_sql("""SELECT Contract,COUNT(*)
            FROM customers
            GROUP BY Contract
            ORDER BY COUNT(*) DESC
            LIMIT 1""",conn)

,Contract,COUNT(*)
0,Month-to-month,3875


### Insights

The most commonly used contract type among customers is Month-to-month, with 3875 customers. This indicates that most customers prefer this contract duration over other available options.

-----------------------------
### Business Question 4:

What is the average monthly charge across all customers?

In [9]:
pd.read_sql(""" SELECT AVG([Monthly Charges])
            FROM customers
            """,conn)

,AVG([Monthly Charges])
0,64.761692


### Insight

The average monthly charge is approximately $64.76, indicating the typical revenue generated per customer each month.

-------------------------------

### Business Question 5

Which payment method is most commonly used?

In [10]:
pd.read_sql(""" SELECT [Payment Method],COUNT(*)
            FROM customers
            GROUP BY [Payment Method]
            ORDER BY COUNT(*) DESC
            LIMIT 1
            """,conn)

,Payment Method,COUNT(*)
0,Electronic check,2365


### Insight

Electronic check is the most frequently used payment method among customers.

----------------------------

### Business Question 6

What is the average tenure for churned and non-churned customers?

In [11]:
pd.read_sql(""" SELECT [Churn Label],AVG([Tenure Months])
                FROM customers
                GROUP BY [Churn Label]
            """,conn)

,Churn Label,AVG([Tenure Months])
0,No,37.569965
1,Yes,17.979133


### Insight

Customers who remain with the company generally have a higher average tenure than customers who churn.

------------------------------

### Business Question 7

How does Customer Lifetime Value (CLTV) vary across contract types?

In [12]:
pd.read_sql(""" SELECT [Contract],AVG(CLTV)
            FROM customers
            GROUP BY [Contract]
        """,conn)

,Contract,AVG(CLTV)
0,Month-to-month,4136.708387
1,One year,4529.955872
2,Two year,4890.214159


### Insight

Long-term contracts tend to generate higher customer lifetime value compared to month-to-month subscriptions.

----------------------------------

### Business Question 8

Which cities have the highest number of customers?

In [13]:
pd.read_sql("""SELECT CITY,COUNT(*)
            FROM customers
            GROUP BY CITY
            ORDER BY COUNT(*) DESC
            """,conn)

,City,COUNT(*)
0,Los Angeles,305
1,San Diego,150
2,San Jose,112
3,Sacramento,108
4,San Francisco,104
...,...,...
1124,Ahwahnee,4
1125,Aguanga,4
1126,Adin,4
1127,Acton,4


### Insight

Customer distribution is concentrated in a few cities, which may represent important markets for the business.

---------------------------------

### Business Question 9

What is the distribution of internet service types?

In [14]:
pd.read_sql(""" SELECT [Internet Service],COUNT(*)
            FROM customers
            GROUP BY [Internet Service]  
            ORDER BY COUNT(*) DESC 
        """,conn)

,Internet Service,COUNT(*)
0,Fiber optic,3096
1,DSL,2421
2,No,1526


### Insight
Fiber Optic is the dominant internet service category within the customer base.

-----------------

### Business Question 10

Which contract type has the highest churn rate?

In [15]:
pd.read_sql("""SELECT Contract,
       COUNT(*) AS total_customers,
       SUM(CASE WHEN [Churn Label] = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
       ROUND(
           100.0 * SUM(CASE WHEN [Churn Label] = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),
           2
       ) AS churn_rate
FROM customers
GROUP BY Contract
ORDER BY churn_rate DESC""",conn)

,Contract,total_customers,churned_customers,churn_rate
0,Month-to-month,3875,1655,42.71
1,One year,1473,166,11.27
2,Two year,1695,48,2.83


### Insight

Month-to-month contracts exhibit the highest churn rate among all contract types. Customers on flexible monthly plans are significantly more likely to leave the company compared to customers with one-year or two-year contracts. In contrast, long-term contracts demonstrate stronger customer retention and lower churn rates.

This suggests that contract duration is an important factor influencing customer loyalty and retention.

----------------------------------------------

### Business Implication

The company should focus on encouraging month-to-month customers to transition to longer-term contracts through loyalty programs, discounts, or bundled service offerings. Increasing long-term contract adoption may help reduce customer churn and improve customer lifetime value.

---------------------------
# Advanced SQL Concepts

## Common Table Expressions (CTE)

### Objective

Use a CTE to simplify complex queries and improve readability.

### Business Question: 

Which contract types have the highest average monthly charges?

In [16]:
pd.read_sql("""WITH contract_stats AS
                ( SELECT Contract,AVG([Monthly Charges])
                  FROM customers
                  GROUP BY Contract
                  ORDER BY AVG([Monthly Charges]) DESC
            )
            SELECT * 
            FROM contract_stats""",conn)

,Contract,AVG([Monthly Charges])
0,Month-to-month,66.398490
1,One year,65.048608
2,Two year,60.770413


The CTE was used to calculate average monthly charges for each contract type. This approach improves query readability by separating the aggregation logic from the final result.

--------------------------------

## Window Function: ROW_NUMBER()

### Objective

Assign a unique rank to customers based on Total Charges.

In [17]:
pd.read_sql("""SELECT CustomerID,
       [Total Charges],
       ROW_NUMBER() OVER(
           ORDER BY [Total Charges] DESC
       ) AS customer_rank
        FROM customers""",conn)

,CustomerID,Total Charges,customer_rank
0,2889-FPWRM,8684.80,1
1,7569-NMZYQ,8672.45,2
2,9739-JLPQJ,8670.10,3
3,9788-HNGUT,8594.40,4
4,8879-XUAHX,8564.75,5
...,...,...,...
7038,3213-VVOLG,0.00,7039
7039,2520-SGTTA,0.00,7040
7040,2923-ARZLG,0.00,7041
7041,4075-WKNIU,0.00,7042


ROW_NUMBER() assigns a unique sequential rank to each customer based on total spending, allowing identification of the highest-value customers.

------------------------------

## Window Function: RANK()

### Objective

Rank customers according to Customer Lifetime Value (CLTV).

In [18]:

pd.read_sql("""SELECT CustomerID,
       CLTV,
       RANK() OVER(
           ORDER BY CLTV DESC
       ) AS cltv_rank
        FROM customers""",conn)

,CustomerID,CLTV,cltv_rank
0,7622-FWGEW,6500,1
1,6024-RUGGH,6499,2
2,0383-CLDDA,6499,2
3,2683-JXWQQ,6495,4
4,8894-JVDCV,6494,5
...,...,...,...
7038,0247-SLUJI,2004,7038
7039,4657-FWVFY,2004,7038
7040,4925-LMHOK,2003,7041
7041,0871-URUWO,2003,7041


RANK() was used to identify customers with the highest lifetime value. Customers with similar CLTV values receive the same ranking position.

-----------------------

## SQL Views

### Objective

Create a reusable view for churned customers.

In [19]:

conn.execute("""
DROP VIEW IF EXISTS churned_customers
""")

conn.execute("""
CREATE VIEW churned_customers AS
SELECT *
FROM customers
WHERE [Churn Label] = 'Yes'
""")


In [20]:
pd.read_sql(""" SELECT *
                FROM churned_customers
                LIMIT 5
            """, conn)

,CustomerID,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,...,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,No,...,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,86,3239,Competitor made better offer
1,9237-HQITU,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,Yes,...,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,67,2701,Moved
2,9305-CDSKC,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,Yes,...,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,Yes,86,5372,Moved
3,7892-POOKP,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,Yes,...,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,84,5003,Moved
4,0280-XJGEX,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,Yes,...,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,Yes,89,5340,Competitor had better devices


### Observation

A SQL view named `churned_customers` was created to store all customers who have churned. Views improve query reusability and simplify future analysis by avoiding repeated filtering conditions.

---------------------------------------

# Python and SQL Integration

## Objective

To integrate Python with SQLite and execute SQL queries directly from Jupyter Notebook using pandas.read_sql().

In [21]:
query = """
SELECT Contract,
       COUNT(*) AS total_customers
FROM customers
GROUP BY Contract
"""

pd.read_sql(query, conn)

,Contract,total_customers
0,Month-to-month,3875
1,One year,1473
2,Two year,1695


Python successfully executed SQL queries and returned results as a pandas DataFrame.

-------------------

In [22]:
query = """
SELECT [Churn Label],
       AVG([Monthly Charges]) AS avg_charge
FROM customers
GROUP BY [Churn Label]
"""

pd.read_sql(query, conn)

,Churn Label,avg_charge
0,No,61.265124
1,Yes,74.441332


In [23]:
query = """
SELECT Contract,
       AVG(CLTV) AS avg_cltv
FROM customers
GROUP BY Contract
"""

pd.read_sql(query, conn)

,Contract,avg_cltv
0,Month-to-month,4136.708387
1,One year,4529.955872
2,Two year,4890.214159


In [24]:
conn.close()

--------------------
# Key Findings

1. The dataset contains 7043 customer records.

2. Month-to-month contracts are the most commonly used subscription type.

3. Electronic check is the preferred payment method among customers.

4. Customers with longer tenure generally have higher Total Charges and CLTV.

5. Month-to-month contracts exhibit the highest churn rate, making them the most at-risk customer segment.


-----------------------------------

# Conclusion

SQL was successfully used to extract, filter, aggregate, and analyze customer data from the Telco Churn dataset. Business-oriented queries helped uncover customer behavior patterns, churn trends, and revenue-related insights. Advanced SQL concepts such as CTEs, window functions, and views improved query organization and analytical capabilities. Integration with Python enabled automated data retrieval and supports future analytics and machine learning workflows.